# 🫀 퀘스트 46 · Q4 — **방법 B: burden 을 특징으로** (층② 눈금)

| | **MedKOS / `notebooks/quest46_q4_burden_feature.ipynb`** |
|---|---|
| 퀘스트 | `ailab-2026-0046` — 층② 점수 눈금 |
| 부모 런 | `quest46_q3_prior_em`(`20260804T1226`) · `quest46_q3b_prior_shuffle`(`20260804T1254`) |
| 사전등록 기준 | Q3 대비 **전역 PR-AUC > 0** · **매크로 비열등** |

## Q3 갈래가 남긴 것

```
C2 ✅   오라클 사전확률 보정이 전역 PR-AUC 를 0.3362 → 0.5514 로 올린다(매크로 0.5389 수준)
B1 ✅   그 이득은 **자기 사전확률 정렬**의 몫이다 — 셔플하면 무너진다(shuf−raw −0.0111)
C3 ⚠️  그런데 **EM 은 실패했다** (Δ −0.0185 · 회복률 −0.086)
        기전: 지배 레코드 48(π* 0.5764 · TEST S 의 34.4%)에서 π̂ 이 clip 바닥 0.0100
        ★ EM 수렴 문제가 아니다 — 평균사후 0.0393 · em@platt 0.0100 · bbse 0.0100.
          **보정된 사후확률 자체가 「이 기록엔 S 가 거의 없다」고 말한다**
```

즉 **사전확률 정렬은 진짜 이득인데, 그걸 무라벨로 추정하는 데서 막혔다.** 방법 B 는 다른
경로를 시도한다 — 점수를 사후에 옮기는 대신 **모델이 레코드 맥락을 직접 쓰게** 한다.

## ★★★ 설계 전 반드시 풀어야 하는 긴장

**선형 모델에 burden 을 「상수 특징」으로 넣으면 그건 레코드별 상수 로짓 시프트다 →
방법 A 와 구조적으로 같다.** 계수를 데이터가 정한다는 차이뿐이고, 레코드 **내** 순위는
그대로다. 합성 실측:

```
raw    전역 0.5946 · 매크로 0.5203
B_add  전역 0.7364 · 매크로 0.5202   ← 매크로 Δ **−0.000122** = 사실상 항등
B_int  전역 0.7367 · 매크로 0.5204   ← 상호작용이라야 레코드 내 순위가 바뀐다
```

그대로 두면 Q4 는 **Q3 를 계수만 바꿔 다시 재는 런**이 된다. 그래서:

- **두 판본을 둘 다 팔로 둔다** — `B_add`(상수 특징)와 `B_int`(burden × 리듬 상호작용)
- **주 관문은 `B_int` 대 `A`** 다. `B_add` 는 **A 와 같음을 보이는 구조 대조**로 쓴다
- 그리고 Q3 와 달리 **매크로가 진짜 판정 대상이다** — `B_int` 는 레코드 내 순위를 바꾸므로
  Q3 의 「매크로는 항등 대조」 논리가 **여기선 안 통한다**

## burden 을 어디서 얻나 — 오라클과 배포 가능판을 나눈다

`burden` 은 라벨에서 나온다. Q7-AA 카드가 정리했듯 임상에서 PAC 부담은 홀터 판독으로
사전에 아는 정보지만, **자동 파이프라인 안에서는 추정해야 한다**.

| 팔 | burden 출처 | 성격 |
|---|---|---|
| `*_oracle` | 진짜 유병률 `π*_r` | **상한** — 배포 불가 |
| `*_em` | Q3 의 EM 추정 `π̂_r` | **배포 가능판** |
| `B_int_shuf` | `π*` 를 레코드끼리 **셔플** | ★ Q3-B 의 대조를 이식 |

## 관문 (사전등록)

| 관문 | 무엇 | 통과 기준 |
|---|---|---|
| **D0** | 재현 — Q3 의 raw·oracle·매크로 | 코호트 동일 시 \|Δ\| ≤ 0.005, 아니면 **리허설** |
| **D1 ★★ 구조** | `B_add` 가 A 와 같은가 · `B_int` 는 다른가 | 관문 아님. **A 와 B 를 가르는 것이 무엇인지**를 구성으로 보인다 |
| **D2 ★★★ 주 관문** | `B_int_oracle − A_oracle` 전역 PR-AUC | 짝지은 차의 CI 가 0 을 뗄 것(사전등록 「Q3 대비 > 0」) |
| **D3** | 배포 가능판 `B_int_em − A_em` | 참고. Q3 에서 `A_em` 이 이미 음수였다 |
| **D4 ★★** | 셔플 대조 `B_int_oracle − B_int_shuf` | 이득이 **자기 burden 정렬**의 몫인가(Q3-B 이식) |
| **D5 ★★ 매크로** | `B_int_oracle` 매크로 vs raw | ★ Q3 와 달리 **진짜 판정 대상**이다(사전등록 「비열등」) |
| **D6** | 결론 검산표 | R38 ⑦ · R39 ⑤ |

### 판정표

- **D2 ✅ · D4 ✅ · D5 ✅** → 방법 B 가 A 를 이긴다. 남은 건 **`π̂` 를 고치는 것**(Q3 의 병목)
- **D2 ❌/미결** → B 가 A 를 못 이긴다. 층② 의 처방은 **A 계열**이고 병목은 여전히 추정이다
- **D4 ❌** → 이득이 burden **정렬**이 아니라 주입 자체에서 온다 → 지표 재검토

⚠️ **새 데이터 0** — `svdb_data5.npz` 만. Q3 와 **같은 분할·같은 기저·같은 보정 절차**를 밟는다.


In [ ]:
# CELL 0 — 공용 사전점검
import numpy as np

def decide(lo, hi, thr, direction):
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if direction == ">":
        if lo > thr: return "✅ 지지"
        if hi < thr: return "❌ 기각"
    else:
        if hi < thr: return "✅ 지지"
        if lo > thr: return "❌ 기각"
    return "⚠️ 미결"

def mde(lo, hi):
    return (hi - lo) / 2.0 if np.isfinite(lo) and np.isfinite(hi) else float("nan")

def boot_mean(v, seed, nb=3000, q=2.5):
    d = np.asarray(v, float); d = d[np.isfinite(d)]
    if len(d) < 3:
        return float("nan"), float("nan"), float("nan"), len(d)
    rng = np.random.RandomState(seed)
    b = [d[rng.randint(0, len(d), len(d))].mean() for _ in range(nb)]
    return (float(d.mean()), float(np.percentile(b, q)),
            float(np.percentile(b, 100 - q)), len(d))

def _rank_avg(v):
    v = np.asarray(v, float); o = v.argsort()
    r = np.empty(len(v), float); r[o] = np.arange(len(v), dtype=float)
    for u in np.unique(v):
        m = v == u
        if m.sum() > 1:
            r[m] = r[m].mean()
    return r

def spearman(a, b):
    """★★ 동점을 **평균 순위**로(Q3-B 에서 argsort 판본의 순서 의존이 드러났다)."""
    ra, rb = _rank_avg(a), _rank_avg(b)
    if np.std(ra) < 1e-12 or np.std(rb) < 1e-12:
        return float("nan")
    return float(np.corrcoef(ra, rb)[0, 1])

def need_super(n, half, eff, p80=False):
    if not np.isfinite(half) or abs(eff) < 1e-9 or n < 1:
        return float("nan")
    r = float(n) * (half / abs(eff)) ** 2
    return r * 2.04 if p80 else r

def ece_of(p, y, nbin=15):
    p = np.asarray(p, float); y = np.asarray(y, float)
    edges = np.linspace(0.0, 1.0, nbin + 1); e = 0.0
    for i in range(nbin):
        m = (p >= edges[i]) & (p < edges[i + 1] if i < nbin - 1 else p <= edges[i + 1])
        if m.any():
            e += m.mean() * abs(p[m].mean() - y[m].mean())
    return float(e)

def derangement(n, rng):
    for _ in range(1000):
        p = rng.permutation(n)
        if not np.any(p == np.arange(n)):
            return p
    return np.roll(np.arange(n), 1)

class AssetError(RuntimeError): pass
print("CELL 0 ✅")


In [ ]:
# CELL 1 — 설정 · 사전등록
import os, sys, json, importlib, time, warnings
importlib.invalidate_caches(); warnings.filterwarnings("ignore")

SMOKE = os.environ.get("MEDKOS_SMOKE") == "1"
_ENV_ROOT = os.environ.get("MEDKOS_DRIVE_ROOT")
if _ENV_ROOT:
    DRIVE_ROOT = _ENV_ROOT
else:
    try:
        from google.colab import drive; drive.mount("/content/drive", force_remount=False)
        DRIVE_ROOT = "/content/drive/MyDrive"
    except Exception as e:
        print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
MITBIH  = os.path.join(DRIVE_ROOT, "mitbih")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

SEED0, IDX_S = 20260804, 1
FS = 360
FULL_K = tuple(range(4, 33))
RHY_K  = (5, 10, 20, 32)
MIN_S, MIN_N = 25, 25

# ── ★ 사전등록 상수 (SMOKE 가 절대 안 건드린다)
TOL_IDENT = 1e-12
TOL_REPRO = 0.005
NONINF_MARGIN = 0.01     # ★ D5 매크로 **비열등 여유**(사전등록 문구가 안 밝혀서 여기서 못 박는다)
SPLIT_PATTERN = ("TRAIN", "TEST", "DEV", "TRAIN", "TEST", "TRAIN", "DEV", "TEST",
                 "TRAIN", "TEST", "DEV", "TRAIN", "TEST", "TRAIN", "DEV", "TEST",
                 "TRAIN", "TEST", "DEV", "TRAIN")

# ── 비용 손잡이
NB_BOOT = 400 if SMOKE else 2000
N_SHUF  = 3   if SMOKE else 10
N_PERM  = 3   if SMOKE else 20     # 대비의 영점(라벨 치환) — Q3-B 의 교훈

ARMS = ("raw", "A_oracle", "A_em", "B_add_oracle", "B_int_oracle", "B_int_em", "B_int_shuf")
READ_ORDER = ("D0", "D1", "D2", "D3", "D4", "D5", "D6")

SV5 = os.path.join(MITBIH, "svdb_data5.npz")

REF = dict(raw=0.3362, oracle=0.5514, em=0.3177,
           macro_prauc=0.538912, macro_auroc=0.871891,
           pi_tr=0.08074, n_train=22, n_dev=14, n_test=20, dominant=0.344,
           dom_rec=48, dom_pi=0.5764, dom_pihat=0.0100,
           c2_gain=0.2151, c3_gain=-0.0185, b1=0.2262, b1_shuf=-0.0111)

RULE_CHECK = {
    "R11 / R11-b":      "전역이 주 지표인 예외 런 — 지배 지분·제외·GMIN_S 병기 · 단독 인용 금지",
    "R16 fallback 없음": "자산 없으면 **중단**",
    "R22 누출 없음":     "기저·보정·EM 하이퍼를 **DEV 에서만** 고정",
    "R26 / R38 ②":      "★★ **대비의 영점**을 측정한다 — 0 을 가정하지 않는다(Q3-B 교훈)",
    "R29 ② 분기 금지":   "D0 이 깨지면 아래를 **안 읽는다**",
    "R33 ① MDE":        "관문마다 MDE. **미결 ≠ 등가**",
    "R34 ③ 대조":       "★★ `B_int_shuf` 는 같은 기저·같은 값 집합, **대응만** 깨진다",
    "R35 ① 자 먼저":    "★★★ `B_add` 로 **A 와 B 를 가르는 것이 무엇인지**를 먼저 보인다",
    "R36 ⑤ 성분 병기":  "차는 성분과 함께만",
    "R38 ⑦ 요약 정합":  "요약·검산표·판정이 같은 갈래여야 한다",
    "R39 ① 여유 고정":  "★ 사전등록이 안 밝힌 **매크로 비열등 여유**를 여기서 못 박는다",
}

CONFIG = dict(
    exp="quest46_q4_burden_feature", quest="ailab-2026-0046", step="burden-feature",
    parent_exp=["quest46_q3_prior_em", "quest46_q3b_prior_shuffle"],
    purpose=("**방법 B — 기록 burden 을 별도 특징으로.** Q3 갈래가 남긴 건 「사전확률 정렬은 "
             "진짜 이득인데(오라클 +0.2151 · 셔플 대조 통과) 무라벨 추정에서 막혔다」이고, "
             "병목은 지배 레코드 48(π* 0.5764)에서 **보정된 사후확률 자체가 틀린 것**이었다. "
             "방법 B 는 점수를 사후에 옮기는 대신 **모델이 레코드 맥락을 직접 쓰게** 한다. "
             "★★★ 그런데 선형 모델에 burden 을 **상수 특징**으로 넣으면 그건 레코드별 상수 "
             "로짓 시프트라 **방법 A 와 구조적으로 같다**(합성 실측 매크로 Δ −0.000122). "
             "**상호작용(burden × 리듬)** 이라야 레코드 **내** 순위가 바뀐다. 그래서 두 판본을 "
             "둘 다 팔로 두고, 주 관문은 **`B_int` 대 `A`** 로, `B_add` 는 **구조 대조**로 쓴다. "
             "★★ 그리고 Q3 와 달리 **매크로가 진짜 판정 대상**이다 — `B_int` 는 레코드 내 순위를 "
             "바꾸므로 Q3 의 「매크로는 항등 대조」 논리가 여기선 안 통한다."),
    dataset="SVDB — svdb_data5.npz (리듬 특징만 · 새 데이터 0)",
    arms=list(ARMS), read_order=READ_ORDER, noninf_margin=NONINF_MARGIN,
    n_boot=NB_BOOT, n_shuf=N_SHUF, n_perm=N_PERM, smoke=SMOKE,
    ref=REF, rule_check=RULE_CHECK,
    predictions={
        "D0": "Q3 의 raw·oracle·매크로 재현. **코호트가 같을 때만** 앵커를 적용하고, "
              "다르면 스스로 **리허설**이라 선언한다(스모크 플래그로 관문을 끄지 않는다)",
        "D1": "★★ **구조 대조(관문 아님).** `B_add_oracle` 의 매크로가 raw 와 **거의 같고** "
              "`B_int_oracle` 은 **다른가**. 이게 A 와 B 를 가르는 것이고, 안 보이면 Q4 는 "
              "Q3 를 계수만 바꿔 다시 재는 런이 된다",
        "D2": "★★★ **주 관문 — `B_int_oracle − A_oracle`** 전역 PR-AUC 짝지은 차(레코드 군집 "
              "부트스트랩). 사전등록 「Q3 대비 > 0」. ★ **대비의 영점을 측정**해 그 기준으로 "
              "판정한다 — Q3-B 에서 대비의 영점이 0 이 아닐 수 있음이 드러났다",
        "D3": "배포 가능판 `B_int_em − A_em`. ⚠️ Q3 에서 `A_em` 이 이미 음수(Δ −0.0185)였으므로 "
              "여기서 결론이 안 날 수 있다 — 그러면 그렇게 쓴다",
        "D4": "★★ **셔플 대조** `B_int_oracle − B_int_shuf`. 같은 기저·같은 burden 값 집합, "
              "**대응만** 깨진다 → 이득이 **자기 burden 정렬**의 몫인지 가른다(Q3-B 이식)",
        "D5": f"★★ **매크로 — 진짜 판정 대상.** `B_int_oracle` 의 매크로가 raw 대비 비열등"
              f"(여유 {NONINF_MARGIN}). 사전등록 문구가 여유를 안 밝혀 **여기서 못 박는다**(R39 ①)",
        "D6": "결론 검산표"},
    caveat=("★★ **오라클 팔은 상한이지 방법이 아니다** — TEST 유병률을 쓰므로 배포 불가. "
            "배포 가능판은 `*_em` 이고 Q3 의 π̂ 를 그대로 물려받는다(그 π̂ 가 지배 레코드에서 "
            "붕괴한 게 Q3 의 결론이다). ★ **매크로 비열등 여유**를 사전등록이 안 밝혔으므로 "
            f"{NONINF_MARGIN} 으로 못 박는다 — 사후에 고르지 않는다(R39 ①). "
            "★ Q3 와 **같은 분할·기저·보정 절차**를 밟는다 — 안 그러면 「Q3 대비」가 성립 안 한다."))
np.random.seed(SEED0)
run = MedKOSRun("quest46_q4_burden_feature", CONFIG, project=PROJECT)
run.log("설정 ✅ **Q4 — 방법 B: burden 을 특징으로**")
run.log("  ★★★ 주 관문 = **`B_int_oracle − A_oracle`** (상호작용 대 로짓 시프트)")
run.log("  ★★ `B_add`(상수 특징)는 **A 와 구조적으로 같다** — D1 이 그걸 보인다")
run.log("  ★★ Q3 와 달리 **매크로가 진짜 판정 대상**이다(레코드 내 순위가 바뀐다)")
run.log(f"  ★ 매크로 비열등 여유 **{NONINF_MARGIN}** — 사전등록이 안 밝혀 여기서 고정(R39 ①)")
if SMOKE:
    run.log(f"  ⚠️ **스모크런** — 비용 손잡이만 축소(NB_BOOT={NB_BOOT} · N_SHUF={N_SHUF} · "
            f"N_PERM={N_PERM}). 관문 문턱은 그대로다")
run.log("\n  사전등록 규칙 체크리스트 (R29 ③)")
for k_, v_ in RULE_CHECK.items():
    run.log(f"    [x] {k_:<18} {v_}")


In [ ]:
# CELL 2 — 【D-0】 코호트 · 분할 · 기저 · DEV 전용 보정 (Q3 과 **같은 절차**)
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import average_precision_score, roc_auc_score

run.log("\n" + "=" * 100)
run.log("【D-0】 코호트 · 분할 · 기저 · DEV 전용 보정 — Q3 과 같은 결정 절차")
run.log("=" * 100)
VERD, DIFF = {}, {}
def g_(k, v, d):
    VERD[k] = v; run.log(f"  {k:<5}{v}  {d}")

if not os.path.exists(SV5):
    raise AssetError(f"{SV5} 없음(R16)")
D5 = np.load(SV5, allow_pickle=True)
PID = np.asarray(D5["pid"]).astype(int); Y3 = np.asarray(D5["y3"]).astype(int)
PRE = np.asarray(D5["pre_rr"], float); POST = np.asarray(D5["post_rr"], float)
K = np.where(Y3 >= 0)[0]
RID = PID[K]; Y = Y3[K]; TT = (Y == IDX_S)
pre = PRE[K].astype(float); post = POST[K].astype(float)
RS = np.array(sorted(set(RID.tolist())))

_S = pd.Series(pre); _G = _S.groupby(pd.Series(RID))
def local_base(k):
    r = np.asarray(_G.apply(lambda x: x.shift(1).rolling(k, min_periods=1).median())).astype(float)
    return np.where(np.isfinite(r), r, pre)
_med = _G.transform("median").to_numpy()
_std = _G.transform("std").to_numpy(); _mean = _G.transform("mean").to_numpy()
f1 = _med - pre
f2 = {k: 1.0 - pre / (local_base(k) + 1e-9) for k in FULL_K}
f3 = post - pre
f4 = np.nan_to_num(_std / (_mean + 1e-9))
RHY = np.nan_to_num(np.c_[f1, np.column_stack([f2[k] for k in RHY_K]), f3, f4,
                          np.log1p(np.clip(pre, 0, None)), np.log1p(np.clip(post, 0, None))],
                    nan=0.0, posinf=0.0, neginf=0.0)

IDXS_ALL = {r: np.where(RID == r)[0] for r in RS}
REC_OK = [r for r in RS
          if TT[IDXS_ALL[r]].sum() >= MIN_S and (~TT[IDXS_ALL[r]]).sum() >= MIN_N]
EXCL = [int(r) for r in RS if r not in REC_OK]
BURD_ALL = {int(r): float(TT[IDXS_ALL[r]].mean()) for r in RS}
order = sorted(REC_OK, key=lambda r: (BURD_ALL[int(r)], int(r)))
SPLIT = {int(r): SPLIT_PATTERN[i % len(SPLIT_PATTERN)] for i, r in enumerate(order)}
TR_R = [r for r in REC_OK if SPLIT[int(r)] == "TRAIN"]
DV_R = [r for r in REC_OK if SPLIT[int(r)] == "DEV"]
TE_R = [r for r in REC_OK if SPLIT[int(r)] == "TEST"]
sel = lambda rs: np.concatenate([IDXS_ALL[r] for r in rs])
TR_I, DV_I, TE_I = sel(TR_R), sel(DV_R), sel(TE_R)
run.log(f"  레코드 {len(RS)} · 채점 가능 {len(REC_OK)} · 제외 {len(EXCL)}")
for nm, rr, ii in (("TRAIN", TR_R, TR_I), ("DEV", DV_R, DV_I), ("TEST", TE_R, TE_I)):
    bs = [BURD_ALL[int(r)] for r in rr]
    run.log(f"  {nm:<6}레코드 {len(rr):>3} · 비트 {len(ii):>7,} · S 유병률 {TT[ii].mean():.4f} "
            f"(레코드별 {min(bs):.4f}~{max(bs):.4f})")
s_cnt = np.array([int(TT[IDXS_ALL[r]].sum()) for r in TE_R], float)
DOMINANT = float(s_cnt.max() / s_cnt.sum()); DOM_REC = int(TE_R[int(np.argmax(s_cnt))])
run.log(f"  ★ TEST 지배 지분 **{DOMINANT:.3f}** — 레코드 {DOM_REC} · **전역 단독 인용 금지**(R11)")

# ── 기저·보정 (Q3 과 동일)
def make_platt(s, y):
    lr = LogisticRegression(max_iter=3000, C=1e6)
    lr.fit(np.asarray(s).reshape(-1, 1), np.asarray(y).astype(int))
    return lambda v: lr.predict_proba(np.asarray(v).reshape(-1, 1))[:, 1]

def make_iso(s, y):
    ir = IsotonicRegression(out_of_bounds="clip", y_min=1e-6, y_max=1 - 1e-6)
    ir.fit(np.asarray(s), np.asarray(y).astype(float))
    return lambda v: np.clip(ir.predict(np.asarray(v)), 1e-6, 1 - 1e-6)

def fit_feats(Ftr, ytr):
    """★ 표준화·적합 모두 **TRAIN 에서만**. 팔마다 특징이 다르므로 팔마다 다시 적합한다."""
    mu, sd = Ftr.mean(0), Ftr.std(0) + 1e-9
    lr = LogisticRegression(max_iter=3000, C=1.0)
    lr.fit((Ftr - mu) / sd, ytr)
    return lambda F: lr.decision_function((F - mu) / sd)

score_raw = fit_feats(RHY[TR_I], TT[TR_I].astype(int))
SC_dev, SC_te = score_raw(RHY[DV_I]), score_raw(RHY[TE_I])
half = len(DV_R) // 2
in_ = lambda rs: np.isin(RID[DV_I], np.asarray(rs))
m_fit, m_sel = in_(DV_R[:half]), in_(DV_R[half:])
CAND = {}
for nm, mk in (("platt", make_platt), ("isotonic", make_iso)):
    f_ = mk(SC_dev[m_fit], TT[DV_I][m_fit])
    CAND[nm] = ece_of(f_(SC_dev[m_sel]), TT[DV_I][m_sel].astype(float))
CAL_NAME = min(CAND, key=lambda k: CAND[k])
calib = {"platt": make_platt, "isotonic": make_iso}[CAL_NAME](SC_dev, TT[DV_I])
EPS = 1e-6
P_dev = np.clip(calib(SC_dev), EPS, 1 - EPS); P_te = np.clip(calib(SC_te), EPS, 1 - EPS)
PI_TR = float(TT[DV_I].mean())
run.log(f"  보정기 {CAL_NAME}(DEV 반쪽 홀드아웃 ECE {CAND[CAL_NAME]:.5f}) · π_tr {PI_TR:.5f} "
        f"(Q3 {REF['pi_tr']})")
CONFIG["cohort"] = dict(n_ok=len(REC_OK), excluded=EXCL, dominant=DOMINANT, dom_rec=DOM_REC,
                        n_train=len(TR_R), n_dev=len(DV_R), n_test=len(TE_R),
                        pi_tr=PI_TR, calibrator=CAL_NAME)
run.save_json("config", CONFIG)


In [ ]:
# CELL 3 — 【D-A】 D0 재현 · 팔 구성 (A = 로짓 시프트 · B = 특징)
run.log("\n" + "=" * 100)
run.log("【D-A】 D0 — Q3 재현 · 팔 구성")
run.log("=" * 100)
RID_te = RID[TE_I]; Y_te = TT[TE_I].astype(int)
TE_POS = {int(r): np.where(RID_te == r)[0] for r in TE_R}
PI_STAR = {int(r): float(Y_te[TE_POS[int(r)]].mean()) for r in TE_R}

def logit(p):
    p = np.clip(np.asarray(p, float), 1e-12, 1 - 1e-12)
    return np.log(p) - np.log1p(-p)

def shift_logit(pi, pi_tr):
    return 0.0 if pi == pi_tr else float(logit(pi) - logit(pi_tr))

L_raw = logit(P_te)
def apply_shift(pi_by):
    out = L_raw.copy()
    for r in TE_R:
        pos = TE_POS[int(r)]
        out[pos] = out[pos] + shift_logit(pi_by[int(r)], PI_TR)
    return out

def em_prior(p, pi_tr, iters=100, tol=1e-9, clip=1e-2):
    pi = float(pi_tr)
    for _ in range(int(iters)):
        w = pi / pi_tr; v = (1.0 - pi) / (1.0 - pi_tr)
        num = w * p
        pp = num / (num + v * (1.0 - p))
        new = float(np.clip(pp.mean(), clip, 1.0 - clip))
        if abs(new - pi) < tol:
            pi = new; break
        pi = new
    return pi
PI_HAT = {int(r): em_prior(P_te[TE_POS[int(r)]], PI_TR) for r in TE_R}

pooled = lambda L: float(average_precision_score(Y_te, L))
def per_record(L):
    ap, au = {}, {}
    for r in TE_R:
        pos = TE_POS[int(r)]; yy = Y_te[pos]
        if 0 < yy.sum() < len(yy):
            ap[int(r)] = float(average_precision_score(yy, L[pos]))
            au[int(r)] = float(roc_auc_score(yy, L[pos]))
    return ap, au
macro_of = lambda L: float(np.mean(list(per_record(L)[0].values())))

AP_RAW = pooled(L_raw)
L_A_or = apply_shift(PI_STAR); L_A_em = apply_shift(PI_HAT)
MACRO_RAW = macro_of(L_raw)
run.log(f"  재현 — raw {AP_RAW:.4f} (Q3 {REF['raw']}) · A_oracle {pooled(L_A_or):.4f} "
        f"(Q3 {REF['oracle']}) · A_em {pooled(L_A_em):.4f} (Q3 {REF['em']})")
run.log(f"         매크로 {MACRO_RAW:.6f} (Q3 {REF['macro_prauc']})")
COHORT_MATCH = (len(RS) == 78 and len(REC_OK) == 56 and len(TE_R) == REF["n_test"]
                and len(DV_R) == REF["n_dev"] and len(TR_R) == REF["n_train"])
d_rep = max(abs(AP_RAW - REF["raw"]), abs(pooled(L_A_or) - REF["oracle"]))
if COHORT_MATCH:
    run.log(f"  ★ 코호트가 Q3 과 같다 → **재현 앵커 적용**(허용 {TOL_REPRO})")
    if d_rep > TOL_REPRO:
        raise AssetError(f"D0 재현 실패(|Δ| {d_rep:.4f}) — 「Q3 대비」가 성립하지 않는다")
else:
    run.log(f"  ⚠️⚠️ **코호트가 다르다** → 재현 앵커 미적용. 이 실행은 **리허설**이다 — 수치 인용 금지")

# ── ★★ B 팔 — burden 을 **특징**으로. 상수 추가 vs 상호작용
def burden_vec(idx, by_rec):
    return np.array([by_rec[int(r)] for r in RID[idx]], float)

BURD_TR = {int(r): BURD_ALL[int(r)] for r in TR_R}      # 학습 레코드의 진짜 유병률
BURD_DV = {int(r): BURD_ALL[int(r)] for r in DV_R}

def _feats(idx, bvec, mode):
    if mode == "add":
        return np.c_[RHY[idx], bvec]
    return np.c_[RHY[idx], bvec, RHY[idx] * bvec[:, None]]

def build_B(by_tr, by_te, mode, ret_fn=False):
    """★ `add` 는 burden 을 **상수 특징**으로, `int` 는 **burden × 리듬 상호작용**으로.

    ⚠️ 「`B_add` 가 raw 의 레코드 내 순위를 정확히 보존한다」는 **틀린 말**이다 — 특징을
    추가하면 로지스틱이 **모든 계수를 다시 적합**하므로 리듬 가중치도 바뀐다.
    정확한 명제는 **「burden 에서 나오는 기여분이 레코드 안에서 상수다」** 이고,
    D1 이 그걸 **직접** 잰다(burden 을 상수로 바꿨을 때의 점수 차의 레코드 내 산포)."""
    btr, bte = burden_vec(TR_I, by_tr), burden_vec(TE_I, by_te)
    fn = fit_feats(_feats(TR_I, btr, mode), TT[TR_I].astype(int))
    sc = fn(_feats(TE_I, bte, mode))
    if ret_fn:
        return sc, (lambda bv: fn(_feats(TE_I, bv, mode)))
    return sc

rng_s = np.random.RandomState(SEED0 + 300)
TE_KEYS = [int(r) for r in TE_R]
SHUF_MAPS = []
for s_ in range(N_SHUF):
    perm = derangement(len(TE_KEYS), np.random.RandomState(SEED0 + 300 + s_))
    SHUF_MAPS.append({TE_KEYS[i]: PI_STAR[TE_KEYS[perm[i]]] for i in range(len(TE_KEYS))})

L = {"raw": L_raw, "A_oracle": L_A_or, "A_em": L_A_em,
     "B_add_oracle": build_B(BURD_TR, PI_STAR, "add"),
     "B_int_oracle": build_B(BURD_TR, PI_STAR, "int"),
     "B_int_em": build_B(BURD_TR, PI_HAT, "int")}
L_shuf = [build_B(BURD_TR, m, "int") for m in SHUF_MAPS]
L["B_int_shuf"] = L_shuf[0]
run.log(f"\n  {'팔':<14}{'전역 PR-AUC':>13}{'매크로':>10}")
POOL, MACRO = {}, {}
for a in ARMS:
    POOL[a] = pooled(L[a]); MACRO[a] = macro_of(L[a])
    run.log(f"  {a:<14}{POOL[a]:>13.4f}{MACRO[a]:>10.4f}")
g_("D0", "✅ 지지" if COHORT_MATCH else "⚠️ 리허설",
   f"Q3 재현(|Δ| {d_rep:.4f}) · 7팔 구성 완료" if COHORT_MATCH else
   "코호트가 달라 재현 앵커가 없다 — 리허설")
CONFIG["D0"] = dict(cohort_match=bool(COHORT_MATCH), repro_delta=float(d_rep),
                    pooled=POOL, macro=MACRO)
run.save_json("config", CONFIG)


In [ ]:
# CELL 4 — 【D-B】 ★★ D1 — 구조 대조: A 와 B 를 가르는 건 무엇인가
run.log("\n" + "=" * 100)
run.log("【D-B】 D1 — **구조 대조**(관문 아님): `B_add` 는 A 와 같고 `B_int` 는 다른가")
run.log("=" * 100)
run.log("  ▸ 선형 모델에서 burden 을 **상수 특징**으로 넣으면 레코드별 **상수 로짓 시프트**다")
run.log("    → 방법 A 와 구조적으로 같다. 계수를 데이터가 정한다는 차이뿐이다")
run.log("  ▸ **상호작용**이라야 레코드 **내** 순위가 바뀐다 — 그때 비로소 B 가 A 와 다른 방법이다")

d_add = abs(MACRO["B_add_oracle"] - MACRO["raw"])
d_int = abs(MACRO["B_int_oracle"] - MACRO["raw"])
run.log(f"\n  매크로 |Δ vs raw| — `B_add` **{d_add:.6f}** · `B_int` **{d_int:.6f}**")
# ★★★ 판정은 **burden 기여분의 레코드 내 산포**로 한다 — 이게 구성으로 정확하다.
#    (「`B_add` 가 raw 의 순위를 보존한다」는 틀린 말이다: 특징을 더하면 로지스틱이 모든
#     계수를 다시 적합하므로 리듬 가중치도 바뀐다. 정확한 명제는 **burden 에서 나오는
#     기여분이 레코드 안에서 상수**라는 것이고, 그건 직접 잴 수 있다.)
run.log("\n  ★★ burden **기여분**의 레코드 내 산포 — burden 을 상수로 바꿨을 때의 점수 차")
B_CONTRIB = {}
for a, mode in (("B_add_oracle", "add"), ("B_int_oracle", "int")):
    _, fn = build_B(BURD_TR, PI_STAR, mode, ret_fn=True)
    b_real = burden_vec(TE_I, PI_STAR)
    b_flat = np.full(len(TE_I), float(np.mean(list(BURD_TR.values()))))
    delta = fn(b_real) - fn(b_flat)                 # ★ burden 에서 온 기여분
    within = float(np.mean([delta[TE_POS[int(r)]].std() for r in TE_R]))
    total = float(delta.std())
    B_CONTRIB[a] = dict(within_sd=within, total_sd=total,
                        ratio=float(within / (total + 1e-12)))
    run.log(f"    {a:<14} 레코드 내 SD **{within:.6e}** / 전체 SD {total:.6e} "
            f"= **{within/(total+1e-12):.4f}**  "
            + ("(0 이면 **레코드별 상수** = A 와 같은 구조)" if within / (total + 1e-12) < 0.01
               else "← **레코드 안에서 변한다**"))
RHO_IN = {}
for a in ("A_oracle", "B_add_oracle", "B_int_oracle"):
    RHO_IN[a] = float(np.mean([spearman(L_raw[TE_POS[int(r)]], L[a][TE_POS[int(r)]])
                               for r in TE_R]))
    run.log(f"    (참고) {a:<14} raw 대비 레코드 내 ρ {RHO_IN[a]:.6f}")
STRUCT_OK = bool(B_CONTRIB["B_add_oracle"]["ratio"] < 0.01
                 and B_CONTRIB["B_int_oracle"]["ratio"] > 0.01)
g_("D1", "✅ 지지" if STRUCT_OK else "⚠️ 미결",
   f"`B_add` 의 burden 기여분은 **레코드별 상수**(내부/전체 SD "
   f"{B_CONTRIB['B_add_oracle']['ratio']:.4f} = A 와 같은 구조)이고 `B_int` 는 레코드 안에서 "
   f"변한다({B_CONTRIB['B_int_oracle']['ratio']:.4f}) — **A 와 B 가 구조적으로 갈린다**"
   if STRUCT_OK else
   f"★★ 구조가 안 갈린다(`B_add` {B_CONTRIB['B_add_oracle']['ratio']:.4f} · "
   f"`B_int` {B_CONTRIB['B_int_oracle']['ratio']:.4f}) — 그러면 Q4 는 **Q3 를 계수만 바꿔 "
   "다시 재는 런**이고, D2 를 「방법 B 가 낫다」로 읽으면 안 된다")
CONFIG["D1"] = dict(macro_delta_add=float(d_add), macro_delta_int=float(d_int),
                    rho_within=RHO_IN, contrib=B_CONTRIB, struct_ok=bool(STRUCT_OK))
run.save_json("config", CONFIG)


In [ ]:
# CELL 5 — 【D-C】 ★★★ D2 주 관문 · D3 배포판 · D4 셔플 대조
run.log("\n" + "=" * 100)
run.log("【D-C】 D2 — **주 관문** `B_int_oracle − A_oracle` · D3 · D4")
run.log("=" * 100)

def cluster_boot(arms, seed, nb):
    """★ 재표집 단위는 **레코드**. 모든 팔을 **같은 재표집**에 태운다(짝지은 차)."""
    rng = np.random.RandomState(seed)
    recs = [int(r) for r in TE_R]
    out = {k: [] for k in arms}
    for _ in range(nb):
        pick = [recs[i] for i in rng.randint(0, len(recs), len(recs))]
        pos = np.concatenate([TE_POS[r] for r in pick])
        yy = Y_te[pos]
        if yy.sum() < 5 or yy.sum() == len(yy):
            continue
        for k, LL in arms.items():
            out[k].append(float(average_precision_score(yy, LL[pos])))
    return {k: np.asarray(v, float) for k, v in out.items()}

T0 = time.time()
BOOTA = dict(L)
for i, LL in enumerate(L_shuf):
    BOOTA[f"shuf{i}"] = LL
B = cluster_boot(BOOTA, SEED0 + 11, NB_BOOT)
SH = np.mean([B[f"shuf{i}"] for i in range(N_SHUF)], axis=0)
run.log(f"  ({time.time()-T0:.0f}초) 부트스트랩 {len(B['raw'])}회 × 팔 {len(BOOTA)}개")

def paired(a, b, name):
    d = B[b] - B[a]
    pt = POOL[b] - POOL[a]
    lo, hi = float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))
    run.log(f"    {name:<28}**{pt:+.4f}** [{lo:+.4f}, {hi:+.4f}] · MDE {mde(lo,hi):.4f}")
    return dict(mean=pt, lo=lo, hi=hi, mde=float(mde(lo, hi)))

run.log("\n  ★★★ 짝지은 차 (같은 부트스트랩 재표집)")
D2 = paired("A_oracle", "B_int_oracle", "D2  B_int_oracle − A_oracle")
D2b = paired("A_oracle", "B_add_oracle", "    B_add_oracle − A_oracle (구조 대조)")
D3 = paired("A_em", "B_int_em", "D3  B_int_em − A_em")
d4 = B["B_int_oracle"] - SH
D4 = dict(mean=POOL["B_int_oracle"] - float(np.mean([POOL["B_int_shuf"]])),
          lo=float(np.percentile(d4, 2.5)), hi=float(np.percentile(d4, 97.5)))
D4["mde"] = float(mde(D4["lo"], D4["hi"]))
run.log(f"    {'D4  B_int_oracle − B_int_shuf':<28}**{D4['mean']:+.4f}** "
        f"[{D4['lo']:+.4f}, {D4['hi']:+.4f}] · MDE {D4['mde']:.4f}")
run.log(f"    ▸ 성분 — raw {POOL['raw']:.4f} · A_oracle {POOL['A_oracle']:.4f} · "
        f"B_int_oracle {POOL['B_int_oracle']:.4f} (R36 ⑤)")

# ── ★★ 대비의 영점 (Q3-B 교훈 — 0 을 가정하지 않는다)
run.log(f"\n  ★★ **대비의 영점** — 학습 라벨 치환 뒤 같은 짝지은 차 (reps={N_PERM})")
nd = []
for s_ in range(N_PERM):
    rr = np.random.RandomState(SEED0 + 400 + s_)
    yperm = TT[TR_I].astype(int)[rr.permutation(len(TR_I))]
    sc_ = fit_feats(RHY[TR_I], yperm)
    pd_, pt_ = sc_(RHY[DV_I]), sc_(RHY[TE_I])
    cal_ = {"platt": make_platt, "isotonic": make_iso}[CAL_NAME](pd_, TT[DV_I])
    q_te = np.clip(cal_(pt_), EPS, 1 - EPS)
    l0 = logit(q_te); pi_n = float(TT[DV_I].mean())
    a_or = l0.copy()
    for r in TE_R:
        p_ = TE_POS[int(r)]
        a_or[p_] = a_or[p_] + shift_logit(PI_STAR[int(r)], pi_n)
    btr = burden_vec(TR_I, BURD_TR); bte = burden_vec(TE_I, PI_STAR)
    Ftr = np.c_[RHY[TR_I], btr, RHY[TR_I] * btr[:, None]]
    Fte = np.c_[RHY[TE_I], bte, RHY[TE_I] * bte[:, None]]
    b_int = fit_feats(Ftr, yperm)(Fte)
    nd.append(pooled(b_int) - pooled(a_or))
nm_, nlo, nhi, nn = boot_mean(nd, SEED0 + 61, NB_BOOT)
run.log(f"    `B_int_oracle − A_oracle` 영점 **{nm_:+.4f}** [{nlo:+.4f}, {nhi:+.4f}] · n={nn}")

# ★★★ 문턱은 **max(0, 영점 상단)** 이다.
#     사전등록 기준이 「Q3 대비 전역 PR-AUC **> 0**」이므로 0 은 반드시 넘어야 하고,
#     Q3-B 교훈대로 **측정된 영점**이 0 보다 높으면 그쪽이 더 엄한 문턱이 된다.
#     영점만 문턱으로 쓰면 영점이 **음수**일 때 효과가 사실상 0 이어도 통과한다
#     (스모크 실측: D2 +0.0002 [−0.0037,+0.0056] 인데 ✅ 로 찍혔다).
D2_THR = max(0.0, nhi) if np.isfinite(nhi) else 0.0
run.log(f"    ▸ D2 문턱 = max(0, 영점 상단 {nhi:+.4f}) = **{D2_THR:+.4f}** "
        "(사전등록 「> 0」 + 영점 안전망을 둘 다 건다)")
d2_v = decide(D2["lo"], D2["hi"], D2_THR, ">")
g_("D2", d2_v,
   f"방법 B(상호작용)가 방법 A 를 **{D2['mean']:+.4f}** 앞선다 (문턱 {D2_THR:+.4f} 초과)"
   if d2_v.startswith("✅") else
   (f"B 가 A 를 **못 이긴다** — 층② 의 처방은 **A 계열**이고 병목은 여전히 **추정**이다"
    if d2_v.startswith("❌") else
    f"미결 — **등가가 아니다**(상한 {D2['hi']:+.4f} · MDE {D2['mde']:.4f})"))
d4_v = decide(D4["lo"], D4["hi"], 0.0, ">")
g_("D4", d4_v,
   "이득이 **자기 burden 정렬**의 몫이다 — 대응을 깨면 무너진다" if d4_v.startswith("✅") else
   "★★ 대응을 깨도 이득이 그대로다 — burden **주입 자체**가 정체다(Q3-B 와 같은 함정)")
CONFIG["D2"] = D2; CONFIG["D2b"] = D2b; CONFIG["D3"] = D3; CONFIG["D4"] = D4
CONFIG["D2_null"] = dict(mean=nm_, lo=nlo, hi=nhi, n=int(nn), thr=float(D2_THR))
run.save_json("config", CONFIG)


In [ ]:
# CELL 6 — 【D-D】 ★★ D5 매크로(진짜 판정) · 진단 · D6 검산표
run.log("\n" + "=" * 100)
run.log("【D-D】 D5 — **매크로(진짜 판정 대상)** · 지배 레코드 진단 · D6 검산표")
run.log("=" * 100)
run.log("  ▸ ★ Q3 에서는 매크로가 **항등 대조**였다(상수 시프트라 정의상 불변).")
run.log("    Q4 의 `B_int` 는 레코드 **내** 순위를 바꾸므로 **매크로가 진짜 판정 대상**이다")

per_raw = per_record(L_raw)[0]
MACd = {}
for a in ("B_add_oracle", "B_int_oracle", "B_int_em"):
    pa = per_record(L[a])[0]
    dv = [pa[r] - per_raw[r] for r in per_raw if r in pa]
    m_, lo_, hi_, n_ = boot_mean(dv, SEED0 + 71, NB_BOOT)
    MACd[a] = dict(mean=m_, lo=lo_, hi=hi_, n=int(n_))
    run.log(f"  {a:<14} 매크로 Δ vs raw **{m_:+.6f}** [{lo_:+.6f}, {hi_:+.6f}] · n={n_}")
d5 = MACd["B_int_oracle"]
d5_v = decide(d5["lo"], d5["hi"], -NONINF_MARGIN, ">")
g_("D5", d5_v,
   f"매크로 비열등 — Δ {d5['mean']:+.6f} 의 CI 하단이 여유 −{NONINF_MARGIN} 위다"
   if d5_v.startswith("✅") else
   f"★★ 매크로가 **열등**하다(Δ {d5['mean']:+.6f}) — 전역을 올리며 레코드 내 순위를 망쳤다")

# ── 진단(관문 아님) — Q3 의 병목 레코드에서 B 가 A 보다 나은가
run.log(f"\n  ★ 진단 — Q3 의 병목이던 지배 레코드 {DOM_REC}(π* {PI_STAR.get(DOM_REC, float('nan')):.4f} · "
        f"π̂ {PI_HAT.get(DOM_REC, float('nan')):.4f})")
if DOM_REC in TE_POS:
    pos = TE_POS[DOM_REC]; yy = Y_te[pos]
    run.log(f"  {'팔':<14}{'그 레코드 PR-AUC':>18}")
    for a in ARMS:
        run.log(f"  {a:<14}{average_precision_score(yy, L[a][pos]):>18.4f}")
    run.log("    ▸ 레코드 **내** PR-AUC 이므로 `raw`·`A_*`·`B_add` 는 같아야 한다(상수 시프트)")

run.log("\n  ★ D6 — **결론 검산표**")
CHECK = [
    dict(claim=f"D1 구조 — `B_add` 매크로Δ {CONFIG['D1']['macro_delta_add']:.6f} vs "
               f"`B_int` {CONFIG['D1']['macro_delta_int']:.6f}",
         num="선형 모델에서 burden 상수 특징 = 레코드별 상수 로짓 시프트(= 방법 A)",
         assume="**없음** — 선형성의 성질이다",
         iffalse="`B_int` 도 안 움직이면 Q4 는 **Q3 를 계수만 바꿔 다시 재는 런**이다"),
    dict(claim=f"D2 주 관문 {D2['mean']:+.4f} [{D2['lo']:+.4f}, {D2['hi']:+.4f}]",
         num=f"영점 {CONFIG['D2_null']['mean']:+.4f} "
             f"[{CONFIG['D2_null']['lo']:+.4f}, {CONFIG['D2_null']['hi']:+.4f}] · "
             f"구조 대조 `B_add−A` {D2b['mean']:+.4f}",
         assume="오라클 burden 이 **레코드 안에서 상수**라는 것",
         iffalse="유병률이 기록 안에서 표류하면 오라클조차 상한이 아니다"),
    dict(claim=f"D4 셔플 대조 {D4['mean']:+.4f} [{D4['lo']:+.4f}, {D4['hi']:+.4f}]",
         num=f"같은 기저·같은 burden 값 집합, **대응만** 깨진다 · derangement {N_SHUF}개",
         assume="TEST 유병률이 셔플로 재배치 가능할 만큼 다양하다는 것",
         iffalse="유병률이 다 비슷하면 셔플이 아무것도 안 바꿔 대조가 무력해진다"),
    dict(claim=f"D5 매크로 — Δ {d5['mean']:+.6f} (여유 −{NONINF_MARGIN})",
         num="★ Q3 와 달리 **진짜 판정 대상**이다 — `B_int` 는 레코드 내 순위를 바꾼다",
         assume=f"비열등 여유 {NONINF_MARGIN} — 사전등록이 안 밝혀 **이 런에서 못 박았다**(R39 ①)",
         iffalse="여유를 사후에 고르면 유리한 값을 고르게 된다"),
    dict(claim="배포 가능판은 `*_em` 이고 Q3 의 π̂ 를 물려받는다",
         num=f"D3 `B_int_em − A_em` {D3['mean']:+.4f} [{D3['lo']:+.4f}, {D3['hi']:+.4f}]",
         assume="Q3 의 π̂ 가 이 코호트에서 유효하다는 것",
         iffalse=f"★ Q3 에서 π̂ 는 지배 레코드({REF['dom_rec']})에서 붕괴했다"
                 f"(π* {REF['dom_pi']} → π̂ {REF['dom_pihat']}) — D3 는 그 한계를 물려받는다"),
    dict(claim=f"전역이 주 지표인 예외 런 (지배 지분 {DOMINANT:.3f})",
         num=f"TEST 레코드 {len(TE_R)} · 제외 {len(EXCL)} · GMIN_S = S≥{MIN_S} & N≥{MIN_N}",
         assume="Q3-B 가 셔플 대조로 이 지표를 검증했다는 것(B1 +0.2262 · shuf−raw −0.0111)",
         iffalse="**전역 단독 인용 금지**(R11)"),
]
for i, c in enumerate(CHECK, 1):
    run.log(f"\n  [{i}] **{c['claim']}**")
    run.log(f"      근거   {c['num']}")
    run.log(f"      가정   {c['assume']}")
    run.log(f"      틀리면 {c['iffalse']}")
CONFIG["D5"] = MACd; CONFIG["D6"] = CHECK
run.save_json("config", CONFIG)


In [ ]:
# CELL 7 — 【D-E】 그림 · 요약 · 마무리
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display
fig, ax = plt.subplots(1, 3, figsize=(16.5, 4.6))
EN = {"raw": "raw", "A_oracle": "A oracle", "A_em": "A em", "B_add_oracle": "B add",
      "B_int_oracle": "B int", "B_int_em": "B int em", "B_int_shuf": "B int shuf"}
xs = np.arange(len(ARMS))
ax[0].bar(xs, [POOL[a] for a in ARMS], color=["tab:gray", "tab:blue", "tab:cyan",
                                              "tab:olive", "tab:red", "tab:orange", "tab:purple"])
ax[0].axhline(MACRO["raw"], ls="--", color="k", lw=1.0, label=f"macro(raw) {MACRO['raw']:.3f}")
ax[0].set_xticks(xs); ax[0].set_xticklabels([EN[a] for a in ARMS], fontsize=7, rotation=20)
ax[0].set_ylabel("pooled PR-AUC")
ax[0].set_title("arms : pooled", fontsize=9)
ax[0].legend(fontsize=7); ax[0].grid(alpha=.3, axis="y")

nm = ["D2  Bint - Aor", "    Badd - Aor", "D3  Bint_em - Aem", "D4  Bint - Bshuf"]
vv = [D2["mean"], D2b["mean"], D3["mean"], D4["mean"]]
lo = [vv[0] - D2["lo"], vv[1] - D2b["lo"], vv[2] - D3["lo"], vv[3] - D4["lo"]]
hi = [D2["hi"] - vv[0], D2b["hi"] - vv[1], D3["hi"] - vv[2], D4["hi"] - vv[3]]
ax[1].errorbar(vv, np.arange(4), xerr=[lo, hi], fmt="o", capsize=5, color="tab:red")
ax[1].axvline(0, color="k", lw=.9)
ax[1].axvline(CONFIG["D2_null"]["hi"], ls=":", color="tab:gray", lw=1.2,
              label=f"D2 null upper {CONFIG['D2_null']['hi']:+.3f}")
ax[1].set_yticks(range(4)); ax[1].set_yticklabels(nm, fontsize=8)
ax[1].set_xlabel("paired difference, pooled PR-AUC")
ax[1].set_title("contrasts", fontsize=9)
ax[1].legend(fontsize=7); ax[1].grid(alpha=.3, axis="x")

mk = ["B_add_oracle", "B_int_oracle", "B_int_em"]
mv = [MACd[a]["mean"] for a in mk]
ml = [mv[i] - MACd[a]["lo"] for i, a in enumerate(mk)]
mh = [MACd[a]["hi"] - mv[i] for i, a in enumerate(mk)]
ax[2].errorbar(mv, np.arange(3), xerr=[ml, mh], fmt="s", capsize=5, color="tab:green")
ax[2].axvline(0, color="k", lw=.9)
ax[2].axvline(-NONINF_MARGIN, ls="--", color="tab:red", lw=1.2,
              label=f"non-inferiority -{NONINF_MARGIN}")
ax[2].set_yticks(range(3)); ax[2].set_yticklabels([EN[a] for a in mk], fontsize=8)
ax[2].set_xlabel("macro PR-AUC delta vs raw")
ax[2].set_title("D5 : macro is a REAL gate here", fontsize=9)
ax[2].legend(fontsize=7); ax[2].grid(alpha=.3, axis="x")
fig.tight_layout()
PNG = run.save_fig("q4_burden_feature", fig)
plt.close(fig); display(Image(PNG))

run.log("\n" + "=" * 100)
run.log("요약")
run.log("=" * 100)
ok_ = lambda k: VERD.get(k, "").startswith("✅")
for g in READ_ORDER[:6]:
    run.log(f"  {g:<5}{VERD.get(g, '(관문 아님)')}")
run.log("")
run.log(f"  전역 PR-AUC — " + " · ".join(f"{a} {POOL[a]:.4f}" for a in ARMS))
run.log(f"  매크로 raw {MACRO['raw']:.4f} · B_int {MACRO['B_int_oracle']:.4f}")
run.log("")
if not CONFIG["D1"]["struct_ok"]:
    run.log("  ⛔ **D1 이 먼저 말한다 — `B_int` 도 레코드 내 순위를 안 바꾼다.**")
    run.log("     그러면 Q4 는 **Q3 를 계수만 바꿔 다시 재는 런**이고, D2 를 「B 가 낫다」로 못 읽는다")
elif ok_("D2") and ok_("D4") and ok_("D5"):
    run.log(f"  ★★★ **방법 B 가 A 를 이긴다** — D2 {D2['mean']:+.4f} "
            f"[{D2['lo']:+.4f}, {D2['hi']:+.4f}] · 셔플 대조 통과 · 매크로 비열등")
    run.log("     → 남은 병목은 **`π̂` 를 고치는 것**이다(Q3 의 결론과 같은 자리)")
elif not ok_("D2"):
    run.log("  ⛔ **B 가 A 를 못 이긴다.**")
    run.log(f"     D2 {D2['mean']:+.4f} [{D2['lo']:+.4f}, {D2['hi']:+.4f}] vs 영점 상단 "
            f"{CONFIG['D2_null']['hi']:+.4f}")
    run.log("     → 층② 의 처방은 **A 계열**이고, 병목은 여전히 **무라벨 사전확률 추정**이다.")
    run.log("       Q3 이 특정한 자리 — 지배 레코드에서 **보정된 사후확률 자체가 틀린다**")
else:
    run.log("  ⚠️ **D2 는 섰지만 셔플 대조 또는 매크로에서 걸렸다** — 아래를 함께 읽는다")
    run.log(f"     D4 {D4['mean']:+.4f} · D5 매크로 Δ {d5['mean']:+.6f}")
run.log("")
run.log(f"  ▸ ★ **전역 단독 인용 금지** — 지배 지분 {DOMINANT:.3f} · 제외 {len(EXCL)} 와 함께만(R11)")
run.log("  ▸ ★ 오라클 팔은 **상한이지 방법이 아니다** — 배포 가능판은 `*_em` 이다")

run.finish({
    "exp_id": "quest46_q4_burden_feature",
    "metric": "b_int_minus_a_oracle",
    "value": float(D2["mean"]),
    "passed": bool(CONFIG["D0"]["cohort_match"] and ok_("D2") and ok_("D4") and ok_("D5")),
    "summary": ("방법 B — burden 을 특징으로. ★ 선형 모델에서 burden 상수 특징은 방법 A 와 "
                "구조적으로 같으므로(레코드별 상수 시프트) 주 관문은 **상호작용 팔** 대 A 로 "
                "잡고, 상수 특징 팔은 구조 대조로 쓴다. Q3 와 달리 매크로가 진짜 판정 대상이다."),
    "verdicts": VERD, "rule_check": RULE_CHECK, "cohort": CONFIG.get("cohort", {}),
    "D0": CONFIG.get("D0", {}), "D1": CONFIG.get("D1", {}), "D2": CONFIG.get("D2", {}),
    "D2b": CONFIG.get("D2b", {}), "D2_null": CONFIG.get("D2_null", {}),
    "D3": CONFIG.get("D3", {}), "D4": CONFIG.get("D4", {}), "D5": CONFIG.get("D5", {}),
    "D6": CONFIG.get("D6", []), "fig": PNG})
run.log(f"\n저장 완료 — {run.dir}")
run.log("다음: `python pipelines/ingest_run.py --results result.json "
        "--notebook notebooks/quest46_q4_burden_feature.ipynb`")
